# Обучение Светланы на её инструментах (LoRA, Colab)

Датасет собирается из живого кода ядра (`node training/build-dataset.mjs`), поэтому модель учит **ровно те** инструменты и схемы, что есть в Светлане.

1. Среда выполнения → GPU (T4 бесплатно хватает на 7B в 4 бит; L4/A100 быстрее).
2. Загрузите `train.jsonl` и `eval.jsonl` (папка `core/training/data`) или клонируйте репозиторий и соберите их здесь.
3. Запустите все ячейки. На T4 ~1 эпоха 3000 примеров ≈ 1–1,5 часа.

In [ ]:
!pip -q install unsloth "trl>=0.12" datasets
# Node нужен только если собираете датасет прямо в Colab
!node -v || (curl -fsSL https://deb.nodesource.com/setup_22.x | bash - && apt-get install -y nodejs)

In [ ]:
# Вариант А: собрать датасет из репозитория (нужен доступ к репо; для приватного — токен)
# !git clone -b svetlana-core-v2 https://github.com/paulafanasyev/Svetlana-2.0 && cd Svetlana-2.0/core && node training/build-dataset.mjs --train 3000 --eval 300
# Вариант Б: загрузить готовые файлы
import os
DATA = 'Svetlana-2.0/core/training/data' if os.path.exists('Svetlana-2.0/core/training/data') else '.'
if not os.path.exists(f'{DATA}/train.jsonl'):
    from google.colab import files; files.upload()  # выберите train.jsonl и eval.jsonl
print(os.listdir(DATA))

In [ ]:
from unsloth import FastLanguageModel
import torch
BASE = 'unsloth/Qwen2.5-7B-Instruct-bnb-4bit'   # шаблон чата Qwen умеет tools; для сервера в РФ потом тот же vLLM
model, tok = FastLanguageModel.from_pretrained(BASE, max_seq_length=8192, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(model, r=16, lora_alpha=32, lora_dropout=0, target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'], use_gradient_checkpointing='unsloth', random_state=7)

In [ ]:
import json
from datasets import Dataset
def norm(msgs):
    out = []
    for m in msgs:
        m = dict(m)
        if m.get('tool_calls'):
            m['tool_calls'] = [{'type':'function','function':{'name':c['function']['name'],'arguments':json.loads(c['function']['arguments'])}} for c in m['tool_calls']]
        out.append(m)
    return out
def load(f):
    rows = [json.loads(l) for l in open(f)]
    return Dataset.from_list([{'text': tok.apply_chat_template(norm(r['messages']), tools=r['tools'], tokenize=False)} for r in rows])
train, evl = load(f'{DATA}/train.jsonl'), load(f'{DATA}/eval.jsonl')
print(len(train), 'примеров; длина первого в токенах:', len(tok(train[0]['text'])['input_ids']))

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only
trainer = SFTTrainer(model=model, tokenizer=tok, train_dataset=train, eval_dataset=evl.select(range(min(100, len(evl)))),
    args=SFTConfig(dataset_text_field='text', max_seq_length=8192, per_device_train_batch_size=1, gradient_accumulation_steps=8, num_train_epochs=1,
                   learning_rate=1e-4, warmup_ratio=0.03, lr_scheduler_type='cosine', logging_steps=10, eval_strategy='steps', eval_steps=100,
                   bf16=torch.cuda.is_bf16_supported(), fp16=not torch.cuda.is_bf16_supported(), optim='adamw_8bit', seed=7, output_dir='out', report_to='none'))
# учим только ответы ассистента (вызовы инструментов и текст), а не системный промпт и результаты инструментов
trainer = train_on_responses_only(trainer, instruction_part='<|im_start|>user\n', response_part='<|im_start|>assistant\n')
trainer.train()

In [ ]:
# Быстрая проверка вживую
FastLanguageModel.for_inference(model)
r = json.loads(open(f'{DATA}/eval.jsonl').readline())
msgs = norm(r['messages'][:2])
ids = tok.apply_chat_template(msgs, tools=r['tools'], add_generation_prompt=True, return_tensors='pt').to('cuda')
print(r['messages'][1]['content'])
print(tok.decode(model.generate(ids, max_new_tokens=200, do_sample=False)[0][ids.shape[1]:]))

In [ ]:
# Сохранение: LoRA (маленькая) + слитая модель для vLLM; при желании GGUF для Ollama/LM Studio
model.save_pretrained('svetlana-lora'); tok.save_pretrained('svetlana-lora')
model.save_pretrained_merged('svetlana-merged', tok, save_method='merged_16bit')
# model.save_pretrained_gguf('svetlana-gguf', tok, quantization_method='q4_k_m')
# model.push_to_hub_merged('ВАШ_НИК/svetlana-7b', tok, save_method='merged_16bit', token='hf_...')
!du -sh svetlana-lora svetlana-merged

## Запуск и экзамен
На GPU-сервере (в РФ: Selectel/Яндекс Облако/Cloud.ru):
```bash
vllm serve ./svetlana-merged --served-model-name svetlana --enable-auto-tool-choice --tool-call-parser hermes --max-model-len 16384
```
Экзамен (из папки `core`):
```bash
node training/eval.mjs --url http://GPU:8000/v1 --model svetlana
```
Вердикт «годится» = ≥90% решений верны и **ни одного** опасного вызова после «отравленной» страницы. Затем во вкладке «ИИ-провайдеры» добавьте preset `vllm`, адрес `http://GPU:8000/v1`, модель `svetlana`.

Сравните с базовой моделью (до обучения) тем же `eval.mjs` — так видно, что обучение реально помогло.